
# FILE domain — release‑aware viewer (v3)

Read‑only audit for `file.csv` with chunk‑safe probes:

- exact header list
- per‑column non‑null counts (DuckDB if available, fallback to chunked pandas estimate)
- distributions for `activity`, `to_removable_media`, `from_removable_media` (exists‑only)
- ID distinct vs dup check (prefers DuckDB; fallback does a sample‑based check)
- top 20 filename bases + extension histogram
- presence of base name `keylogger` (case‑insensitive)
- 50‑row preview with suspected external media usage
- size estimator: write two microscopic parquets (leanish/fullish) and scale up
- final JSON summary: available columns, row estimate, and recommended FULL/LEAN column sets for **this** release

**Guardrails**: this notebook writes only two tiny estimator parquets into `out/<release>/tmp_file_est/`. Everything else is in‑memory.


In [ ]:

# bootstrap
from nb_paths import bootstrap, csv_path
from pathlib import Path
import os, json, math, re
import pandas as pd

env = bootstrap()
REL = env.RELEASE
CSV = str(csv_path(env, "file"))
OUT = Path(env.OUT) / REL
TMP_EST = OUT / "tmp_file_est"
TMP_EST.mkdir(parents=True, exist_ok=True)
print("Release:", REL)
print("CSV exists:", Path(CSV).exists())
print("Estimator tmp:", TMP_EST)


## 1) Header snapshot (exact columns)


In [ ]:

hdr = list(pd.read_csv(CSV, nrows=0, dtype=str).columns)
print("Columns:", hdr)
hdr


## 2) Per‑column non‑null counts (DuckDB preferred, chunked fallback)


In [ ]:
## 2) Per-column non-null counts (DuckDB preferred, chunked fallback)

def non_null_counts_duckdb(csv_path: str, columns: list[str]):
    import duckdb
    con = duckdb.connect(database=":memory:")
    # Read with loose typing; we’ll cast per-column ourselves.
    cols_escaped = ",".join([f'"{c}"' for c in columns])
    con.execute(f"""
        CREATE OR REPLACE TABLE t AS
        SELECT {cols_escaped}
        FROM read_csv_auto('{csv_path}', SAMPLE_SIZE=200000, IGNORE_ERRORS=TRUE);
    """)
    # Count values that are NOT NULL and NOT empty after trim.
    # Cast to VARCHAR before TRIM/NULLIF to avoid BOOL vs '' errors.
    exprs = ",".join([
        f'COUNT(NULLIF(TRIM(CAST("{c}" AS VARCHAR)), "")) AS "{c}"'
        for c in columns
    ])
    row = con.execute(f"SELECT {exprs} FROM t").fetchone()
    return dict(zip(columns, [int(x) for x in row]))

def non_null_counts_chunked(csv_path: str, columns: list[str], chunksize: int = 500_000):
    seen = {c: 0 for c in columns}
    for chunk in pd.read_csv(csv_path, dtype=str, chunksize=chunksize):
        for c in columns:
            if c in chunk.columns:
                s = chunk[c]
                # not-null AND non-empty after strip
                mask = s.notna() & (s.astype(str).str.strip() != "")
                seen[c] += int(mask.sum())   # sum(), not int(series)
    return seen

try:
    import duckdb  # noqa
    nn_counts = non_null_counts_duckdb(CSV, hdr)
    print("Non-null counts computed via DuckDB.")
except Exception as e:
    print(f"DuckDB path failed ({e}); falling back to chunked pandas.")
    nn_counts = non_null_counts_chunked(CSV, hdr)
    print("Non-null counts computed via chunked pandas (fallback).")

nn_counts

## 3) Value distributions (activity, to_removable_media, from_removable_media)


In [ ]:

want_cols = [c for c in ["activity","to_removable_media","from_removable_media"] if c in hdr]
dist = {c: {} for c in want_cols}

def _bump(d, k, n=1):
    if k is None: k = "<NULL>"
    k = str(k)
    d[k] = d.get(k, 0) + n

for chunk in pd.read_csv(CSV, dtype=str, chunksize=500_000):
    for c in want_cols:
        if c in chunk.columns:
            vc = chunk[c].value_counts(dropna=False)
            for k, n in vc.items():
                _bump(dist[c], k if pd.notna(k) else None, int(n))

# show top 12 values per column
summary = {c: dict(sorted(v.items(), key=lambda x: -x[1])[:12]) for c, v in dist.items()}
summary


## 4) ID distinct vs dups check


In [ ]:

id_info = {"id_present": "id" in hdr, "distinct": None, "dups_found": None, "method": None}
if "id" in hdr:
    try:
        import duckdb
        con = duckdb.connect(database=":memory:")
        con.execute(f"CREATE OR REPLACE TABLE t AS SELECT id FROM read_csv_auto('{CSV}', SAMPLE_SIZE=200000, IGNORE_ERRORS=TRUE)")
        total = con.execute("SELECT COUNT(*) FROM t").fetchone()[0]
        ndist = con.execute("SELECT COUNT(DISTINCT id) FROM t").fetchone()[0]
        id_info.update({"distinct": int(ndist), "total_sampled": int(total), "dups_found": (ndist < total), "method": "duckdb_sample"})
    except Exception:
        # fallback: sample first N ids chunked and look for duplicates within that sample
        seen = set(); dup = False; sampled = 0
        for chunk in pd.read_csv(CSV, dtype=str, chunksize=300_000):
            ids = chunk["id"].dropna().astype(str).tolist()
            for x in ids:
                if x in seen:
                    dup = True
                    break
                seen.add(x)
            sampled += len(ids)
            if len(seen) > 1_000_000 or dup:
                break
        id_info.update({"distinct": len(seen), "total_sampled": sampled, "dups_found": dup, "method": "chunked_sample"})
id_info


## 5) Top 20 filename bases + extension breakdown


In [ ]:

import os
from collections import Counter

def base_and_ext(fn: str):
    if not isinstance(fn, str) or fn.strip() == "":
        return ("", "")
    base = os.path.basename(fn).rsplit(".", 1)[0].lower()
    ext  = os.path.basename(fn).split(".")[-1].lower() if "." in os.path.basename(fn) else ""
    return (base, ext)

base_counter = Counter()
ext_counter  = Counter()

if "filename" in hdr:
    for chunk in pd.read_csv(CSV, dtype=str, chunksize=500_000):
        bxe = chunk["filename"].map(base_and_ext)
        for b, e in bxe:
            base_counter[b] += 1
            ext_counter[e]  += 1

top_bases = base_counter.most_common(20)
top_exts  = ext_counter.most_common(20)
{"top_bases": top_bases, "top_exts": top_exts}


## 6) Presence of base name `keylogger`


In [ ]:

keylog_hits = 0
if "filename" in hdr:
    for chunk in pd.read_csv(CSV, dtype=str, chunksize=500_000):
        fn = chunk["filename"].astype(str)
        base = fn.str.extract(r'([^/\\]+?)(?:\.[^.]+)?$', expand=False).str.lower()
        keylog_hits += int((base == "keylogger").sum())
keylog_hits


## 7) Preview 50 rows indicating external media usage


In [ ]:

preview_rows = []
if set(["to_removable_media","from_removable_media"]).intersection(hdr):
    for chunk in pd.read_csv(CSV, dtype=str, chunksize=500_000):
        df = chunk.copy()
        for c in ["to_removable_media","from_removable_media"]:
            if c in df.columns:
                # coerce truthy strings
                df[c] = df[c].astype(str).str.strip().str.lower().isin(["1","true","t","y","yes"])
        mask = False
        if "to_removable_media" in df.columns:
            mask = mask | df["to_removable_media"]
        if "from_removable_media" in df.columns:
            mask = mask | df["from_removable_media"]
        sub = df.loc[mask]
        if not sub.empty:
            cols = [c for c in ["date","id","user","pc","filename","activity","to_removable_media","from_removable_media","content"] if c in sub.columns]
            preview_rows.append(sub[cols].head(50 - sum(len(x) for x in preview_rows)))
            if sum(len(x) for x in preview_rows) >= 50:
                break
pd.concat(preview_rows, ignore_index=True) if preview_rows else pd.DataFrame()


In [ ]:
# === PREVIEW: 50 rows with to/from removable media true (copy/paste friendly)
import pandas as pd

cols = [c for c in ["date","id","user","pc","filename","activity","to_removable_media","from_removable_media","content"] if c in hdr]

rows = []
for chunk in pd.read_csv(CSV, dtype=str, chunksize=500_000):
    df = chunk.copy()
    if "to_removable_media" in df.columns:
        df["to_removable_media"] = df["to_removable_media"].astype(str).str.strip().str.lower().isin(["1","true","t","y","yes"])
    if "from_removable_media" in df.columns:
        df["from_removable_media"] = df["from_removable_media"].astype(str).str.strip().str.lower().isin(["1","true","t","y","yes"])
    mask = False
    if "to_removable_media" in df.columns:   mask = mask | df["to_removable_media"]
    if "from_removable_media" in df.columns: mask = mask | df["from_removable_media"]
    sub = df.loc[mask, cols]
    if not sub.empty:
        rows.append(sub.head(50 - sum(len(x) for x in rows)))
        if sum(len(x) for x in rows) >= 50:
            break

preview = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame(columns=cols)
print(preview.to_csv(index=False, lineterminator="\n"))

In [ ]:
# === LDAP JOIN COVERAGE on ~200k sample (robust; no nrows; PyArrow schema peek)
import json
import pandas as pd
from pathlib import Path

# helpers (fallbacks if your modules aren't importable here)
try:
    from src.helpers.users import normalize_user_series as _norm_user
except Exception:
    def _norm_user(s):
        s = s.astype(str).str.strip()
        s = s.str.replace(r'^[A-Za-z0-9._-]+[\\/]', '', regex=True)  # strip DOMAIN/
        s = s.str.replace(r'@.*$', '', regex=True)                   # strip email domain
        return s.str.lower()

try:
    from src.helpers.time import month_start as _month_start
except Exception:
    def _month_start(x):
        dt = pd.to_datetime(x, errors="coerce", utc=True)
        return pd.to_datetime({'year': dt.dt.year, 'month': dt.dt.month, 'day': 1}).dt.tz_localize('UTC')

# locate LDAP as-of (lean)
ldap_p = Path(env.OUT) / env.RELEASE / "ldap_v3_lean" / "ldap_asof_by_month.parquet"
if not ldap_p.exists():
    print(json.dumps({"error": f"missing {ldap_p}"}))
else:
    # ---- sample ~200k rows from file.csv
    try:
        import duckdb
        con = duckdb.connect(database=":memory:")
        con.execute(f"""
          CREATE OR REPLACE TABLE s AS
          SELECT "date","id","user","pc","filename","activity"
          FROM read_csv_auto('{CSV}', SAMPLE_SIZE=200000, IGNORE_ERRORS=TRUE)
          USING SAMPLE 200000 ROWS
        """)
        samp = con.execute("SELECT * FROM s").df()
        src = "duckdb_rows"
    except Exception as e:
        # pandas fallback: take head from first chunks until ~200k
        target = 200_000
        seen = 0
        bufs = []
        for ch in pd.read_csv(CSV, dtype=str, chunksize=500_000):
            take = min(len(ch), max(1, target - seen))
            bufs.append(ch.head(take))
            seen += take
            if seen >= target:
                break
        samp = pd.concat(bufs, ignore_index=True) if bufs else pd.DataFrame(columns=["date","id","user","pc","filename","activity"])
        src = f"pandas_chunks (no_duckdb): {seen}"

    # derive keys for join
    samp["timestamp"]   = pd.to_datetime(samp["date"], errors="coerce", utc=True)
    samp["event_month"] = _month_start(samp["timestamp"])
    samp["user_key"]    = _norm_user(samp["user"])

    # ---- load only the LDAP columns that actually exist (schema via PyArrow)
    try:
        import pyarrow.parquet as pq
        pf = pq.ParquetFile(str(ldap_p))
        ldap_cols_present = set(pf.schema.names)
    except Exception:
        # if PyArrow is missing for some reason, just read the whole thing (last resort)
        ldap_cols_present = set()

    desired = {"user_key","event_month","email","role","is_admin","team","supervisor_key","first_seen","last_seen"}
    cols_to_read = sorted(desired & ldap_cols_present) if ldap_cols_present else None

    ldap = pd.read_parquet(ldap_p, columns=cols_to_read)
    # normalize types
    if "user_key" in ldap.columns:
        ldap["user_key"] = ldap["user_key"].astype(str)
    if "event_month" in ldap.columns:
        ldap["event_month"] = _month_start(ldap["event_month"])

    prev = samp.merge(ldap, on=["user_key","event_month"], how="left", suffixes=("", "_ldap"))

    n = len(prev)
    got_role = prev["role"].notna() if "role" in prev.columns else pd.Series([False]*n)
    got_email = prev["email"].notna() if "email" in prev.columns else pd.Series([False]*n)
    matched = int((got_role | got_email).sum())
    rate = round(matched / n, 4) if n else 0.0

    out = {
        "source": src,
        "rows_sampled": int(n),
        "matched_role_or_email": matched,
        "match_rate": rate,
        "ldap_cols_present": sorted(list(ldap_cols_present)) if ldap_cols_present else list(ldap.columns),
    }
    print(json.dumps(out, indent=2))

## 8) Rolling size estimator (writes two tiny parquets to `out/<release>/tmp_file_est/`)


In [ ]:

def _pq_kwargs():
    import os
    rg = int(os.getenv("CERT_PARQUET_RG", "512000"))
    comp = os.getenv("CERT_PARQUET_COMP", "zstd")
    try:
        lvl = int(os.getenv("CERT_PARQUET_LEVEL", "3"))
    except Exception:
        lvl = 3
    return dict(engine="pyarrow", index=False, compression=comp, compression_level=lvl, row_group_size=rg)

# choose leanish/fullish column sets that only include present columns
leanish_want = ["date","id","user","pc","filename","activity","to_removable_media","from_removable_media"]
fullish_want = leanish_want + ["content"]
leanish_cols = [c for c in leanish_want if c in hdr]
fullish_cols = [c for c in fullish_want if c in hdr]

leanish_p = TMP_EST / "file_leanish.parquet"
fullish_p = TMP_EST / "file_fullish.parquet"

wrote_any = False
try:
    import duckdb
    con = duckdb.connect(database=":memory:")
    if leanish_cols:
        cols = ",".join([f'"{c}"' for c in leanish_cols])
        con.execute(f"""
            COPY (
              SELECT {cols}
              FROM read_csv_auto('{CSV}', SAMPLE_SIZE=200000, IGNORE_ERRORS=TRUE)
              USING SAMPLE 0.5 PERCENT
            ) TO '{str(leanish_p)}'
            (FORMAT PARQUET, COMPRESSION ZSTD, ROW_GROUP_SIZE 512000);
        """)
        wrote_any = True
    if fullish_cols:
        cols = ",".join([f'"{c}"' for c in fullish_cols])
        con.execute(f"""
            COPY (
              SELECT {cols}
              FROM read_csv_auto('{CSV}', SAMPLE_SIZE=200000, IGNORE_ERRORS=TRUE)
              USING SAMPLE 0.5 PERCENT
            ) TO '{str(fullish_p)}'
            (FORMAT PARQUET, COMPRESSION ZSTD, ROW_GROUP_SIZE 512000);
        """)
        wrote_any = True
    est_src = "duckdb"
except Exception as e:
    # Fallback: build a tiny pandas sample and write with pyarrow
    est_src = f"pandas_fallback: {e}"
    import pandas as _pd
    # Take ~0.5% by grabbing first ~0.5% of rows across first few chunks
    target = 0.005  # 0.5%
    seen = 0
    total = 0
    leanish_buf = []
    fullish_buf = []
    for chunk in _pd.read_csv(CSV, dtype=str, chunksize=500_000):
        total += len(chunk)
        frac_needed = max(0.0, (target*total - seen) / max(len(chunk),1))
        if frac_needed <= 0:  # have enough
            break
        take = max(1, int(len(chunk) * min(frac_needed, 0.1)))  # cap per-chunk to avoid over‑sampling
        samp = chunk.sample(n=take, random_state=42) if len(chunk) > 0 else chunk
        if leanish_cols:
            leanish_buf.append(samp[leanish_cols])
        if fullish_cols:
            fullish_buf.append(samp[fullish_cols])
        seen += len(samp)
        if seen >= target*total:
            break
    if leanish_cols and leanish_buf:
        _pd.concat(leanish_buf, ignore_index=True).to_parquet(leanish_p, **_pq_kwargs())
        wrote_any = True
    if fullish_cols and fullish_buf:
        _pd.concat(fullish_buf, ignore_index=True).to_parquet(fullish_p, **_pq_kwargs())
        wrote_any = True

# Size and scale estimates (bytes)
slean = leanish_p.stat().st_size if leanish_p.exists() else 0
sfull = fullish_p.stat().st_size if fullish_p.exists() else 0
estlean = int(slean * 200 * 12 / 10) if slean else 0  # x200 for 0.5%, +20% headroom
estfull = int(sfull * 200 * 12 / 10) if sfull else 0
{
    "source": est_src,
    "sample_sizes_bytes": {"leanish": slean, "fullish": sfull},
    "estimated_final_bytes": {"lean": estlean, "full": estfull},
}


## 9) Final JSON summary (schema + estimates + recommended columns)


In [ ]:

# Try to estimate total rows with DuckDB; else leave None
def estimate_total_rows(csv_path: str):
    try:
        import duckdb
        con = duckdb.connect(database=":memory:")
        n = con.execute(f"SELECT COUNT(*) FROM read_csv_auto('{csv_path}', SAMPLE_SIZE=200000, IGNORE_ERRORS=TRUE)").fetchone()[0]
        return int(n)
    except Exception:
        return None

row_estimate = estimate_total_rows(CSV)

# Recommended columns based on presence (contract‑aware)
full_want = [
    "timestamp","event_month","user_key","user_raw","pc","filename","activity",
    "to_removable_media","from_removable_media","content","id","date",
    # LDAP context (when present in join during ETL)
    "email","role","is_admin","team","supervisor_key","first_seen","last_seen",
    "employee_name","business_unit","functional_unit","department"
]
lean_want = [
    "timestamp","event_month","user_key","pc","filename","activity","is_Keylogger",
    "to_removable_media","from_removable_media","role","is_admin","team","supervisor_key",
    "first_seen","user_is_active_employee","after_hours"
]

# What actually exists in THIS release's CSV (pre‑join, pre‑derives)
available_now = hdr

rec_full = [c for c in full_want if c in available_now or c in ["timestamp","event_month","user_key","user_raw"]]  # first four are derived
rec_lean = [c for c in lean_want if c in available_now or c in ["timestamp","event_month","user_key","is_Keylogger","after_hours","user_is_active_employee"]]  # derived flags will exist post‑ETL

# Also surface distributions and counts we computed
out = {
    "release": REL,
    "csv_path": CSV,
    "available_columns": available_now,
    "row_estimate": row_estimate,
    "recommended_full_columns": rec_full,
    "recommended_lean_columns": rec_lean,
    "non_null_counts": nn_counts,
    "value_distributions_top": summary,
    "id_check": id_info,
    "keylogger_hits": int(keylog_hits) if 'keylog_hits' in globals() else 0,
    "estimator_outputs": {
        "leanish_parquet": str((OUT / 'tmp_file_est' / 'file_leanish.parquet').relative_to(Path(env.OUT) / REL)),
        "fullish_parquet": str((OUT / 'tmp_file_est' / 'file_fullish.parquet').relative_to(Path(env.OUT) / REL)),
    }
}
print(json.dumps(out, indent=2))
out
